In [1]:
import rawpy
import torch
import numpy as np
from PIL import Image

In [2]:
raw_path = "/hkfs/work/workspace/scratch/ma_sagnihot-projects/2014_000007.nef"

In [3]:
img = rawpy.imread(raw_path)
img = img.raw_image

In [4]:
img.shape

(4012, 6080)

In [5]:
arr = np.expand_dims(img, axis=0) # Needed only if working with a single image, if you have a batch of images then please skip this! 

In [6]:
arr.shape

(1, 4012, 6080)

In [7]:
resized_r = arr[:, ::2,::2]
resized_g1 = arr[:, 0::2, 1::2]
resized_g2 = arr[:, 1::2, 0::2]
resized_b = arr[:, 1::2, 1::2]

In [8]:
resized_r.shape

(1, 2006, 3040)

In [9]:
resized_g1.shape

(1, 2006, 3040)

In [10]:
resized_g2.shape

(1, 2006, 3040)

In [11]:
resized_b.shape

(1, 2006, 3040)

In [12]:
resized_g = np.concatenate((resized_g1, resized_g2), axis=0).mean(axis=0)

In [13]:
resized_g.shape

(2006, 3040)

In [14]:
rgb_image = np.stack((resized_r[0], resized_g, resized_b[0]), axis=0)

In [15]:
rgb_image.shape

(3, 2006, 3040)

In [16]:
print("RAW min: {}\nRAW max: {}".format(rgb_image.min(), rgb_image.max()))

RAW min: 0.0
RAW max: 4095.0


In [30]:
bit_depth = 12 # PASCAL RAW 12-bit, ZURICH 10-bit, RAW-NOD (Sony/Nikon) 14-bit, RAOD 24-bit

In [41]:
rgb_image_normalized = rgb_image/2**bit_depth  # SINCE THE RAW IMAGE HAS DIFFERENT bit depths, depending on the dataset, they need to be normalized between 0 to 1.

In [42]:
print("Normalized min: {}\nNormalized max: {}".format(rgb_image_normalized.min(), rgb_image_normalized.max()))

Normalized min: 0.0
Normalized max: 0.999755859375


# At this point, we have a RGB channel image normalized between 0 to 1. RAW images cannot look good in RGB unless there is some processing, we have 2 papers on this! Below is one such processing, gamma scaling

In [43]:
gamma_scaling = rgb_image_normalized**0.16  # GAMMA SCALING, 0.16 is one possible value (was learnt for the gamma-quant paper, end-to-end with an object detection model)

In [44]:
np.unique(rgb_image_normalized)

array([0.00000000e+00, 1.22070312e-04, 2.44140625e-04, ...,
       9.93652344e-01, 9.96093750e-01, 9.99755859e-01], shape=(5895,))

In [45]:
print("gamma_scaling min: {}\ngamma_scaling max: {}".format(gamma_scaling.min(), gamma_scaling.max()))

gamma_scaling min: 0.0
gamma_scaling max: 0.999960933493968


# However, for direct comparison, RAW --> RGB, we should not be doing Gamma Scaling. However, if we want to use RAW images and learn on them, then we do need Gamma scaling, or even better log-Gamma scaling (Read the paper: https://arxiv.org/pdf/2509.22448?)

In [46]:
rgb_image_scaled_normalized = gamma_scaling * ((2**8)-1) # TO SCALE THE INTENSITIES FROM 0 to 255 (bit depth: 8)

In [47]:
print("gamma_scaling Normalized min: {}\ngamma_scaling Normalized max: {}".format(rgb_image_scaled_normalized.min(), rgb_image_scaled_normalized.max()))

gamma_scaling Normalized min: 0.0
gamma_scaling Normalized max: 254.99003804096182


In [48]:
rgb_image_scaled_normalized = np.einsum('chw->hwc', rgb_image_scaled_normalized) # PUTTING THE CHANNEL DIMENSION LAST SINCE PIL WANTS IT THAT WAY

In [49]:
rgb_pil_image = Image.fromarray(np.uint8(rgb_image_scaled_normalized))

In [50]:
rgb_pil_image.save('RAW_to_RGB.png')